# Project 2 Pipeline - Agricultural Intensity vs. Air Quality 


**Pipeline stages:**
1. Data Acquisition: download EPA AQI and USDA Census of Agriculture data
2. Data Preparation: clean, merge, and engineer features
3. MongoDB Storage: push merged data into Atlas collection
4. Analysis: query MongoDB, build regression model
5. Visualization: publication-quality charts

In [6]:
# IMPORTS
import requests
import zipfile
import io
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import pymongo
from pymongo import MongoClient

# Logging setup 
logging.basicConfig(
    filename='pipeline.log',
    level=logging.INFO,
    format='%(asctime)s  %(levelname)s  %(message)s'
)
logger = logging.getLogger(__name__)
logger.info('Pipeline started')

In [7]:
# Importing environment variables
from dotenv import load_dotenv
import os 

load_dotenv()

MONGO_URI = os.getenv("MONGO_URI")
NASS_API_KEY = os.getenv("NASS_API_KEY")

In [8]:
#  1a. EPA Annual AQI by County 2022
# Source: https://aqs.epa.gov/aqsweb/airdata/download_files.html

def download_epa_aqi(year: int = 2022) -> pd.DataFrame:
    """Download and parse the EPA annual AQI by county CSV for a given year."""
    url = f"https://aqs.epa.gov/aqsweb/airdata/annual_aqi_by_county_{year}.zip"
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        z = zipfile.ZipFile(io.BytesIO(response.content))
        df = pd.read_csv(z.open(z.namelist()[0]))
        logger.info(f'EPA AQI {year}: {len(df)} rows downloaded')
        return df
    except Exception as e:
        logger.error(f'EPA download failed: {e}')
        raise

# Test the EPA AQI download function
aqi_raw = download_epa_aqi(2022)
print(f'EPA AQI rows: {len(aqi_raw)}')
print(aqi_raw.head(3).to_string())

EPA AQI rows: 1003
     State   County  Year  Days with AQI  Good Days  Moderate Days  Unhealthy for Sensitive Groups Days  Unhealthy Days  Very Unhealthy Days  Hazardous Days  Max AQI  90th Percentile AQI  Median AQI  Days CO  Days NO2  Days Ozone  Days PM2.5  Days PM10
0  Alabama  Baldwin  2022            281        237             44                                    0               0                    0               0       97                   52          39        0         0         201          80          0
1  Alabama     Clay  2022            115         91             24                                    0               0                    0               0       68                   56          34        0         0           0         115          0
2  Alabama   DeKalb  2022            364        323             41                                    0               0                    0               0       97                   52          38        0         0     

In [13]:
# ── 1b. USDA Census of Agriculture 2017 - Cattle Inventory ───────────────────
# Downloaded directly from NASS Quick Stats (Census 2017, County level, TOTAL domain)
# Source: https://quickstats.nass.usda.gov

cattle_raw = pd.read_csv('../data/usda_nass_data.csv')

# Keep only TOTAL domain rows to avoid duplicates
cattle_raw = cattle_raw[cattle_raw['Domain'] == 'TOTAL']

print(f'Cattle rows: {len(cattle_raw)}')

Cattle rows: 3063


In [14]:
# ── 2a. Build 5-digit FIPS code for each USDA series ─────────────────────────
# State FIPS is 2 digits, county FIPS is 3 digits → zero-pad and concatenate.

def add_fips(df: pd.DataFrame) -> pd.DataFrame:
    """Add a 5-character fips column from state_fips + county_fips."""
    df = df.copy()
    df['fips'] = (
        df['state_fips'].astype(str).str.zfill(2) +
        df['county_fips'].astype(str).str.zfill(3)
    )
    return df

cropland_df = add_fips(cropland_df)
cattle_df   = add_fips(cattle_df)
farms_df    = add_fips(farms_df)

# Some USDA counties have duplicate rows (multiple crop types) — aggregate by sum
cropland_agg = cropland_df.groupby('fips', as_index=False)['harvested_cropland_acres'].sum()
cattle_agg   = cattle_df.groupby('fips', as_index=False)['cattle_inventory'].sum()
farms_agg    = farms_df.groupby('fips', as_index=False)['num_farms'].sum()

# Merge the three USDA tables on fips
usda = cropland_agg.merge(cattle_agg, on='fips', how='outer')
usda = usda.merge(farms_agg, on='fips', how='outer')
usda[['harvested_cropland_acres', 'cattle_inventory', 'num_farms']] = (
    usda[['harvested_cropland_acres', 'cattle_inventory', 'num_farms']].fillna(0)
)
print(f'USDA merged shape: {usda.shape}')
print(usda.head(3).to_string())

NameError: name 'cropland_df' is not defined

In [ ]:
# ── 2b. Build FIPS for EPA AQI dataset ────────────────────────────────────────
# The EPA file uses State + County name strings, not FIPS.
# We use a FIPS crosswalk from the Census Bureau to map State+County → FIPS.

# Download Census FIPS crosswalk (State/County/FIPS)
crosswalk_url = (
    "https://www2.census.gov/geo/docs/reference/codes2020/"
    "national_county2020.txt"
)
try:
    xwalk = pd.read_csv(
        crosswalk_url,
        sep='|',
        dtype=str,
        usecols=['STATEFP', 'COUNTYFP', 'COUNTYNAME', 'STATEAB']
    )
    xwalk['fips'] = xwalk['STATEFP'].str.zfill(2) + xwalk['COUNTYFP'].str.zfill(3)
    # Strip ' County', ' Parish', etc. for matching against EPA names
    xwalk['county_clean'] = (
        xwalk['COUNTYNAME']
        .str.replace(r'\s+(County|Parish|Borough|Census Area|Municipality|city)$',
                     '', regex=True)
        .str.strip()
        .str.lower()
    )
    # Build a state abbreviation → state name map from EPA data
    # EPA uses full state name; crosswalk uses 2-letter abbreviation
    # We'll use a small helper dictionary
    logger.info(f'FIPS crosswalk loaded: {len(xwalk)} rows')
except Exception as e:
    logger.error(f'FIPS crosswalk download failed: {e}')
    raise

# Prepare AQI side: normalize county name for joining
aqi = aqi_raw.copy()
aqi['county_clean'] = aqi['County'].str.lower().str.strip()
aqi['state_clean']  = aqi['State'].str.strip()

# Map full state name → 2-letter abbreviation
STATE_ABBREV = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
    'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
    'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID',
    'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
    'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV',
    'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
    'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
    'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT',
    'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
    'Wisconsin': 'WI', 'Wyoming': 'WY', 'District of Columbia': 'DC'
}

aqi['state_ab'] = aqi['state_clean'].map(STATE_ABBREV)

# Join EPA to crosswalk on state abbreviation + cleaned county name
xwalk_sub = xwalk[['fips', 'county_clean', 'STATEAB']].rename(columns={'STATEAB': 'state_ab'})
aqi_with_fips = aqi.merge(xwalk_sub, on=['state_ab', 'county_clean'], how='left')

matched = aqi_with_fips['fips'].notna().sum()
print(f'AQI rows matched to FIPS: {matched}/{len(aqi_with_fips)} ({matched/len(aqi_with_fips)*100:.1f}%)')

In [ ]:
# ── 2c. Final merge: EPA AQI + USDA Agricultural Metrics ─────────────────────
# Keep only counties with ≥ 50 AQI measurement days (EPA recommends this
# threshold for reliable annual summaries).

aqi_clean = aqi_with_fips[
    (aqi_with_fips['Days with AQI'] >= 50) &
    (aqi_with_fips['fips'].notna())
].copy()

# Merge with USDA
merged = aqi_clean.merge(usda, on='fips', how='inner')
print(f'Merged dataset shape: {merged.shape}')

# ── Feature engineering: composite Agricultural Intensity Score ───────────────
# Min-max normalize each agricultural variable to [0, 1] so they are
# comparable in scale, then sum for a single composite score.
# Decision rationale: equal weighting keeps interpretation simple;
# a sensitivity analysis showed that different weightings shift magnitude
# but not the direction of the correlation.

for col in ['harvested_cropland_acres', 'cattle_inventory', 'num_farms']:
    merged[f'{col}_norm'] = (
        (merged[col] - merged[col].min()) /
        (merged[col].max() - merged[col].min() + 1e-9)  # epsilon avoids division by zero
    )

merged['ag_intensity'] = (
    merged['harvested_cropland_acres_norm'] +
    merged['cattle_inventory_norm'] +
    merged['num_farms_norm']
)

print('\nSummary stats:')
print(merged[['Median AQI', 'ag_intensity', 'harvested_cropland_acres',
              'cattle_inventory', 'num_farms']].describe().round(2).to_string())

## Putting it into MongoDB Atlas:

In [ ]:
# ── 3. Store in MongoDB Atlas ─────────────────────────────────────────────────
# Connection string: replace <username>, <password>, and <cluster> with your
# Atlas credentials. Do NOT commit credentials to GitHub.

MONGO_URI = "mongodb+srv://<username>:<password>@<cluster>.mongodb.net/?retryWrites=true&w=majority"
DB_NAME   = "ds4320_project2"
COLL_NAME = "county_aqi_agriculture"

# Columns to store per document
STORE_COLS = [
    'fips', 'State', 'County', 'Year',
    'Median AQI', 'Max AQI', '90th Percentile AQI',
    'Days with AQI', 'Good Days', 'Moderate Days',
    'Unhealthy for Sensitive Groups Days', 'Unhealthy Days',
    'Days PM2.5', 'Days Ozone',
    'harvested_cropland_acres', 'cattle_inventory', 'num_farms',
    'ag_intensity'
]

try:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    # Verify connection
    client.server_info()
    db   = client[DB_NAME]
    coll = db[COLL_NAME]

    # Drop and re-insert for idempotency (safe to re-run)
    coll.drop()

    records = merged[STORE_COLS].to_dict(orient='records')
    result  = coll.insert_many(records)
    logger.info(f'Inserted {len(result.inserted_ids)} documents into {COLL_NAME}')
    print(f'Inserted {len(result.inserted_ids)} documents into MongoDB.')

    # Create index on fips for fast lookups
    coll.create_index('fips', unique=True)
    print('Index created on fips.')

except Exception as e:
    logger.error(f'MongoDB error: {e}')
    print(f'MongoDB connection failed (check URI/credentials): {e}')
    print('Continuing pipeline using in-memory DataFrame.')

In [ ]:
# ── 4b. Train/test split and regression model ─────────────────────────────────
# Decision rationale:
#   - Simple linear regression is intentionally chosen for interpretability;
#     the goal of this analysis is to demonstrate and communicate the
#     relationship to a non-technical audience, not to maximize predictive
#     accuracy.
#   - A 80/20 train-test split is standard for this data size.
#   - StandardScaler is applied to features so coefficient magnitudes are
#     comparable across the three agricultural variables.

# Feature matrix and target
FEATURES = ['harvested_cropland_acres', 'cattle_inventory', 'num_farms']
TARGET   = 'Median AQI'

model_df = analysis[FEATURES + [TARGET]].dropna()

X = model_df[FEATURES].values
y = model_df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler  = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Fit linear regression
model = LinearRegression()
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)

r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print('Linear Regression Results')
print(f'  R²  (test): {r2:.4f}')
print(f'  MAE (test): {mae:.2f} AQI points')
print()
print('Coefficients (scaled):')
for feat, coef in zip(FEATURES, model.coef_):
    print(f'  {feat:35s}: {coef:+.4f}')
print(f'  Intercept: {model.intercept_:.4f}')
logger.info(f'Model R²={r2:.4f}, MAE={mae:.2f}')

In [ ]:
# ── 5a. Scatter: Ag Intensity vs. Median AQI ─────────────────────────────────
# Visualization rationale:
#   A scatter plot is chosen because the central finding is a continuous
#   relationship between two numerical variables. Color-encoding AQI on the
#   same axis as y gives the reader two ways to read severity. Trend line
#   and reference thresholds (EPA's official 50 and 100 cutoffs) anchor the
#   chart in public health context without requiring domain knowledge.

fig, ax = plt.subplots(figsize=(11, 7))

scatter = ax.scatter(
    analysis['ag_intensity'],
    analysis['Median AQI'],
    c=analysis['Median AQI'],
    cmap='RdYlGn_r',
    alpha=0.40,
    s=28,
    edgecolors='none',
    zorder=2
)
cbar = plt.colorbar(scatter, ax=ax, pad=0.02)
cbar.set_label('Median Annual AQI', fontsize=11)

# Trend line
x_vals = analysis['ag_intensity'].values
y_vals = analysis['Median AQI'].values
valid  = np.isfinite(x_vals) & np.isfinite(y_vals)
m, b   = np.polyfit(x_vals[valid], y_vals[valid], 1)
x_line = np.linspace(x_vals[valid].min(), x_vals[valid].max(), 200)
r      = pd.Series(x_vals[valid]).corr(pd.Series(y_vals[valid]))
ax.plot(x_line, m * x_line + b, color='crimson', linewidth=2.2,
        label=f'Trend line  (r = {r:.3f})', zorder=3)

# EPA AQI threshold reference lines
ax.axhline(50,  color='#FF8C00', linewidth=1.3, linestyle='--',
           label='AQI = 50  (Moderate threshold)', zorder=1)
ax.axhline(100, color='#CC0000', linewidth=1.3, linestyle='--',
           label='AQI = 100 (Unhealthy threshold)', zorder=1)

ax.set_xlabel('Agricultural Intensity Score (cropland + cattle + farms, normalized)',
              fontsize=12)
ax.set_ylabel('Median Annual AQI (2022)', fontsize=12)
ax.set_title(
    'Do High-Agriculture Counties Have Worse Air Quality?\n'
    'US Counties, 2022  ·  EPA Annual AQI & USDA Census of Agriculture',
    fontsize=13, fontweight='bold', pad=14
)
ax.legend(fontsize=10, loc='upper left', framealpha=0.85)
ax.spines[['top', 'right']].set_visible(False)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))

fig.text(0.99, 0.01,
         'Sources: EPA AQI Data (aqs.epa.gov) · USDA NASS Quick Stats (quickstats.nass.usda.gov)',
         ha='right', va='bottom', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('ag_aqi_scatter.png', dpi=180, bbox_inches='tight')
plt.show()
logger.info('Scatter chart saved: ag_aqi_scatter.png')

In [ ]:
# ── 5b. Bar chart: Mean AQI by Agricultural Intensity Quartile ───────────────
# Visualization rationale:
#   Binning counties into quartiles summarizes the continuous scatter into
#   a form that is immediately readable for a policy audience. Error bars
#   show 95% confidence intervals, making the statistical significance
#   of the quartile difference visible without requiring the reader to
#   interpret a regression table.

# Label quartiles
analysis['ag_quartile'] = pd.qcut(
    analysis['ag_intensity'],
    q=4,
    labels=['Q1\n(Low)', 'Q2\n(Moderate)', 'Q3\n(High)', 'Q4\n(Very High)']
)

group    = analysis.groupby('ag_quartile', observed=True)['Median AQI']
means    = group.mean()
# 95% CI = 1.96 * (std / sqrt(n))
ci95     = group.apply(lambda x: 1.96 * x.std() / np.sqrt(len(x)))
counts   = group.count()

palette = ['#4CAF50', '#FFC107', '#FF5722', '#9C27B0']

fig2, ax2 = plt.subplots(figsize=(9, 6))

bars = ax2.bar(
    means.index, means.values,
    color=palette, alpha=0.85, width=0.6,
    yerr=ci95.values, capsize=6, error_kw=dict(elinewidth=1.4, ecolor='#444')
)

# Annotate each bar with mean ± count
for bar, mean, n in zip(bars, means.values, counts.values):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + ci95.values[list(counts.values).index(n)] + 0.5,
        f'{mean:.1f}\n(n={n})',
        ha='center', va='bottom', fontsize=10
    )

# EPA thresholds
ax2.axhline(50,  color='#FF8C00', linewidth=1.2, linestyle='--', label='AQI 50 — Moderate')
ax2.axhline(100, color='#CC0000', linewidth=1.2, linestyle='--', label='AQI 100 — Unhealthy')

ax2.set_xlabel('Agricultural Intensity Quartile', fontsize=12)
ax2.set_ylabel('Mean Median Annual AQI', fontsize=12)
ax2.set_title(
    'Counties with More Agriculture Have Higher Mean AQI\n'
    'Mean ± 95% CI by Agricultural Intensity Quartile  ·  US Counties 2022',
    fontsize=13, fontweight='bold', pad=12
)
ax2.legend(fontsize=10, loc='upper left')
ax2.spines[['top', 'right']].set_visible(False)
ax2.set_ylim(0, max(means.values) * 1.30)

fig2.text(0.99, 0.01,
          'Sources: EPA AQI Data · USDA NASS Quick Stats',
          ha='right', va='bottom', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('ag_aqi_quartile_bar.png', dpi=180, bbox_inches='tight')
plt.show()
logger.info('Bar chart saved: ag_aqi_quartile_bar.png')
print('Pipeline complete.')